# Preliminary XGBoost Model: Search Conducted

Predicts whether a police stop resulted in a search (`search_conducted`). Uses **stratified K-fold CV** (rare outcome ~4.4%) and **PR-AUC** as the primary metric.

## 1. Setup & Load

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

RANDOM_STATE = 42
N_FOLDS = 5

In [2]:
df = pd.read_csv('./data/sopp_svi_merged.csv')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()

time_parsed = pd.to_datetime(df['time'], errors='coerce')
df['hour'] = time_parsed.dt.hour

print("Shape:", df.shape)
print("Search rate:", df['search_conducted'].eq(True).mean() * 100, "%")

Shape: (383027, 28)
Search rate: 4.252441733872547 %


/var/folders/13/4gcchqp1423dk8lllkh6qns00000gn/T/ipykernel_70780/345155985.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_parsed = pd.to_datetime(df['time'], errors='coerce')


## 2. Preprocessing (aligned with wrangle_data + midterm EDA)

- Complete-case analysis
- Collapse `reason_for_stop` to top 6 + Other (per midterm)
- For XGBoost: keep `hour` as numeric (handles nonlinearity)
- Encode categoricals for XGBoost

In [3]:
# Target
df['search_conducted'] = (df['search_conducted'] == True) | (df['search_conducted'] == 'True')
df['y'] = df['search_conducted'].astype(int)

# Collapse reason_for_stop to top 6 + Other (per midterm EDA)
K = 6
top_reasons = df['reason_for_stop'].value_counts(dropna=True).head(K).index.tolist()
df['reason_for_stop'] = df['reason_for_stop'].where(df['reason_for_stop'].isin(top_reasons), 'Other')

# Modeling columns (complete-case)
model_cols = ['subject_age', 'subject_race', 'subject_sex', 'reason_for_stop', 'service_area',
              'year', 'month', 'day_of_week', 'hour', 'svi_rpl_themes']

mask = df[model_cols + ['y']].notna().all(axis=1)
df_cc = df.loc[mask].copy()

print("Complete cases:", len(df_cc))
print("Search rate (cc):", df_cc['y'].mean() * 100, "%")

Complete cases: 358859
Search rate (cc): 4.422071064122678 %


In [4]:
# Encode categoricals for XGBoost
cat_cols = ['subject_race', 'subject_sex', 'reason_for_stop', 'service_area', 'day_of_week']
encoders = {}

X = df_cc[model_cols].copy()
y = df_cc['y'].values

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

X = X.astype(float)
print("Feature matrix shape:", X.shape)
print("Features:", list(X.columns))

Feature matrix shape: (358859, 10)
Features: ['subject_age', 'subject_race', 'subject_sex', 'reason_for_stop', 'service_area', 'year', 'month', 'day_of_week', 'hour', 'svi_rpl_themes']


## 3. Stratified K-Fold + XGBoost

Stratified K-fold ensures each fold preserves the ~4.4% positive class proportion. PR-AUC is appropriate for imbalanced classification.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y) / max(y.sum(), 1) - 1,  # handle class imbalance
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

# Cross-validated predicted probabilities
y_proba = cross_val_predict(model, X, y, cv=skf, method='predict_proba')[:, 1]

pr_auc = average_precision_score(y, y_proba)
roc_auc = roc_auc_score(y, y_proba)

print("PR-AUC (average precision):", round(pr_auc, 4))
print("ROC-AUC:", round(roc_auc, 4))

/opt/miniconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:43:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/opt/miniconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:43:52] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/opt/miniconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:43:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/opt/miniconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:43:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/opt/miniconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:43:56] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


PR-AUC (average precision): 0.1537
ROC-AUC: 0.7851


In [6]:
# Fit on full data for feature importance
model.fit(X, y)

imp = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature importance:")
print(imp.to_string(index=False))

/opt/miniconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [17:43:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Feature importance:
        feature  importance
reason_for_stop    0.330354
    subject_sex    0.163096
   subject_race    0.109090
 svi_rpl_themes    0.101341
   service_area    0.067640
           hour    0.066257
           year    0.049759
    subject_age    0.048857
          month    0.032271
    day_of_week    0.031335


## 5. XGBoost Slide Content (parallel to Logistic Regression)

**Why:** Handles nonlinearity & interactions, no linear log-odds assumption, strong predictive performance, built-in feature importance.

**Assumptions to Check:** Fewer than logistic—no linearity or conditional independence required. Main considerations: overfitting (controlled via max_depth, regularization) and sufficient data.

**Class Imbalance:** Same as logistic—Precision-Recall AUC, Stratified 5-Fold Cross-Validation, plus `scale_pos_weight` in the model.

**Regularization:** L1 (alpha) and L2 (lambda) on leaf weights, plus max_depth and min_child_weight to limit tree complexity.

In [ ]:
# Partial dependence plots (XGBoost analogue to binned log-odds for logistic)
# Shows marginal effect of each feature on predicted probability — use for XGBoost slide
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

features = ['subject_age', 'hour', 'svi_rpl_themes']
titles = ['Subject Age', 'Hour of Day', 'SVI (Vulnerability Index)']
for ax, feat, title in zip(axes, features, titles):
    idx = list(X.columns).index(feat)
    PartialDependenceDisplay.from_estimator(
        model, X, [idx], ax=ax,
        kind='average', grid_resolution=30
    )
    ax.set_ylabel('Avg predicted P(search)')
    ax.set_title(title)
    ax.set_xlabel(feat)
plt.suptitle('XGBoost: How Features Affect Predicted Search Probability', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('xgboost_pdp.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: xgboost_pdp.png (use for slide)")

## 4. Notes on XGBoost Assumptions

XGBoost has fewer strict assumptions than logistic/mixed-effects models:
- No linearity assumption; handles nonlinear effects and interactions
- No distributional assumptions on residuals
- `scale_pos_weight` helps with class imbalance
- Stratified K-fold + PR-AUC are appropriate for rare outcomes